## Aeronef (NUMPYFILE ring)

In [ ]:
# import sys
# sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

# from FotR_0219 import FRODO, SAM
datafolder = '/home/airbus/CETACEO_cp_interp/DATA/GATALANI_Aero_nef'
db = FRODO(datafolder, format='NUMPYFILE', initial_parse=True, file = ['db_cyc.npy', 'db_random.npy'])

# db = FRODO('/home/airbus/CETACEO_cp_interp/', format = 'NUMPYFILE', initial_parse=True, file=['case_en_npy.npy'])

#FRODO puede llevar distintos "anillos" (los formatos). En este caso llevamos Aeronef, que está organizado mediante archivos binarios de npy. Tenemos dos, y aunque podríamos mezclarlos, vamos a usar solo uno (db_random.npy)
# initial_parse es siempre recomendable activarlo. Estando en True, hacemos un análisis inicial de los datos y completamos sim_metadata, que nos servirá para navegar mejor por la base.

In [ ]:
# SAM tiene muchas subclases útiles. Una de ellas es DictVisualizer, que sirve básicamente para resumir diccionarios. Después tiene Weapons, HDF5Reader y Gardener, que las explicaré en otro momento.

# SAM.DictVisualizer.rich_tree(db.sim_metadata)
#sim_metadata guarda variables, parámetros y distribución de los archivos que has leído. Parecerá una tontería, pero te ayuda a saber qué es lo que tienes. En este caso es sencillo. Cuando la base es propia (como con CODA), hay mucho más guardado.
# Es necesario haber ejecutado db.reader.parse_simulation_dirs()

In [ ]:
# SAM.DictVisualizer.rich_tree(db.npy_dict)
# En el formato NUMPYFILE, los datos en bruto se guardan en npy_dict

In [ ]:
# Estos dos métodos se utilizan para definir data_dict. Este diccionario resumen las variables que se van a utilizar para fabricar el Dataset final.
# ATENCIÓN: Estos dos métodos son generales para todos los formatos, por lo que los inputs pueden cambiar de uno a otro.

db.extract_inputs(
    keys_inputs={
        'ptos': 'db_random.npy/Airfoil', #ptos siempre es obligatorio porque guarda la geometría. En lo demás, importa solo la forma del array. Si coincide con la geometría, será el campo de una variable. Si no, será parámetro e indicará condición de la simulación.
        'aoa': 'db_random.npy/Alpha', # Por ejemplo, podríamos haber puesto aquí los coeficientes cL o cD, para relacionar el campo fluido con las fuerzas.
        'vel': 'db_random.npy/Vinf'
    },
    keys_aux={
        
        },
    method_to_sort='concave_hull', # Es clave ordenar los puntos para los plots y los posibles cálculos de gradientes posteriores. Hay varias formas de ordenar: 'centroid', 'kdtree', 'concave_hull'.
    common = ['ptos']
)

db.extract_outputs(
    keys_outputs={
        'cp': 'db_random.npy/Cp',
        'cf': 'db_random.npy/Cf'
    },
)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(2*6,5))

ax = ax.flatten()
scale = 3
#perfil
sc0 = ax[0].scatter(
    x = db.data_dict['inputs']['ptos'][:, 0],
    y = db.data_dict['inputs']['ptos'][:, 1],
    c='black',
    s=1,
)
ax[0].set_clip_box(ax[0].bbox)
ax[0].set_xlabel('X (m)')
ax[0].set_ylabel('Y (m)')
ax[0].set_ylim([db.data_dict['inputs']['ptos'][:, 1].min()*scale, db.data_dict['inputs']['ptos'][:, 1].max()*scale])
# ax[0].legend(['RAE2822'], loc='upper right')
sc1 = ax[1].scatter(
    x=db.data_dict['inputs']['aoa'],
    y=db.data_dict['inputs']['vel'],
    c='blue',
    marker='o',
    
    # hue=db.data_dict['outputs']['cp'][:, 0],
    # palette='coolwarm'
    )
ax[1].set_xlabel('Angle of Attack (deg)')
ax[1].set_ylabel('Velocity (m/s)')

# Poner anotación en la figura sobre el tipo de perfil (RAE2822) y el número de condiciones de vuelo (1007)
fig.text(0.5, 0.95, 'RAE2822 - 1007 flight conditions', ha='center', fontsize=14)

### Cálculo de derivadas (2D)

In [ ]:
import torch
stencil = 5
polyorder = 2
tensor_ptos = torch.from_numpy(db.data_dict['inputs']['ptos'])
for out in ['cp', 'cf']:
    tensor_out = torch.from_numpy(db.data_dict['outputs'][out])
    mod = torch.zeros(tensor_out.shape, dtype=torch.float64)
    mod2 = torch.zeros(tensor_out.shape, dtype=torch.float64)
    for case in range(tensor_out.shape[-1]):
        mod[:, case] = SAM.Weapons.surface_derivative(
            X=tensor_ptos,
            f=tensor_out[:, case],
            order=1,
            stencil_width=stencil,   
            poly_order=polyorder,
        )
        mod2[:, case] = SAM.Weapons.surface_derivative(
            X=tensor_ptos,
            f=tensor_out[:, case],
            order=2,
            stencil_width=stencil,   
            poly_order=polyorder,
        )
        
    db.sets.add_aux(array_name=f'd{out}_ds', array=mod, notes=f'Modulo de derivada 1 de {out}')
    db.sets.add_aux(array_name=f'd{out}2_ds', array=mod2, notes=f'Modulo de derivada 2 de {out}')


In [ ]:
SAM.DictVisualizer.rich_tree(db.data_dict)

In [ ]:
# Para añadir cálculos indirectos adicionales se utiliza add_aux
import torch
tensor_ptos = torch.from_numpy(db.data_dict['inputs']['ptos'])
for out in ['cp', 'cf']:
    tensor_out = torch.from_numpy(db.data_dict['outputs'][out])

    tensor_deri1 = torch.empty((tensor_out.shape[0], tensor_out.shape[1], 2))
    tensor_deri2 = torch.empty((tensor_out.shape[0], tensor_out.shape[1], 2))
    for case in range(tensor_out.shape[1]):
        tensor_deri1[:, case, :] = SAM.Weapons.finite_diff_derivative(tensor_ptos, tensor_out[:,case], order = 1)
        tensor_deri2[:, case, :] = SAM.Weapons.finite_diff_derivative(tensor_ptos, tensor_out[:,case], order = 2)   

    mod = torch.sqrt(tensor_deri1[:,:,0]**2 + tensor_deri1[:,:,1]**2).reshape_as(tensor_out).numpy()
    mod2 = torch.sqrt(tensor_deri2[:,:,0]**2 + tensor_deri2[:,:,1]**2).reshape_as(tensor_out).numpy()

    db.sets.add_aux(array_name=f'd{out}_ds', array=mod, notes=f'Modulo de derivada 1 de {out}')
    db.sets.add_aux(array_name=f'd{out}2_ds', array=mod2, notes=f'Modulo de derivada 2 de {out}')
    
SAM.DictVisualizer.rich_tree(db.data_dict)

### Cálculo de clusters

In [ ]:
# Por si quieres calcular clusters en la solución: 
df_data_complete, df_metrics = SAM.Weapons.GMM(
    df_data=db.df_data,
    BIC_study=False,
    groupby=["aoa", "vel"],
    nclusters=2,
    features=["dcp_ds", "dcp2_ds"],#, "dcf_ds", "dcf2_ds"],
    save_pictures=True,
    folder_to_save='./GMM_aeronef/',
    n_components_range=range(1, 8),
    covariance_type="diag",
    max_iter=300,
    random_state=123,
    return_metrics_table=True,
    plot_global_analysis=True,
    verbose = True
)
import os, pandas as pd
os.makedirs('./example_GMM_aeronef/')
df_data_complete.to_csv('./example_GMM_aeronef/df_data_complete.csv')
# No hago mucho incapié en esto todavía, pero tiene su miga.

### Exportar datasets

In [ ]:
# Yo tengo mi propio formato de dataset con diccionarios (jset). Este en concreto se guarda en dict_tensors.
db.sets.create_jset(
    sol='all',
    verbose = False,
    save_path = None
)

In [ ]:
# Por supuesto, está SMEAGOL dando su opinión, aunque de momento solo está para el formato CODA.

In [ ]:
import pandas as pd
df_data_complete = pd.read_csv('./example_GMM_aeronef/df_data_complete.csv')
df_data_complete['aoa_vel'] = df_data_complete['aoa'].astype(str) + '_' + df_data_complete['vel'].astype(str)
import os, matplotlib.pyplot as plt
def var_to_features(vars):
    return [f'd{var}_ds' for var in vars]

def plot_case_0(case, vars, df, save:bool = False):
    scale = 7
    markersize_dcp = 2
    features = var_to_features(vars)
    
    
    def get_column_from_df(column, case, df):
        if isinstance(column, (list, tuple)):
            lista = []
            for col in column:
                lista.append(df.groupby(['aoa_vel']).get_group((df['aoa_vel'].unique()[case]))[col])
            return lista
        
        if isinstance(column, str):
            serie = df.groupby(['aoa_vel']).get_group((df['aoa_vel'].unique()[case]))[column]
            return serie
        
    [x, z, cp, clusters] = get_column_from_df(['x', 'z', 'cp', 'clusters_GMM'], case, df)
    list_features_log = get_column_from_df(features, case, df)
    
    fig, ax = plt.subplots(2, 1, figsize=(8, 2*4))
    # ax = ax.flatten()
    ax[0].scatter(
        x, z,
        c='black', s=1
    )
    ax00 = ax[0].twinx()
    ax00.scatter(
        x, list_features_log[0], c='blue', s=markersize_dcp
    )

    ax01 = ax[0].twinx()
    ax01.scatter(
        x, list_features_log[1], c='red', s=markersize_dcp
    )
    ax[0].set_ylim(bottom = z.min()*scale, top = z.max()*scale)
    # Poner un tercer eje a la izquierda con cp
    ax_cp = ax[0].twinx()
    ax_cp.scatter(
        x, cp, c='green', s=markersize_dcp
    )
    ax_cp.spines['left'].set_position(('outward', 60))
    ax_cp.spines['left'].set_color('green')
    ax_cp.tick_params(axis='y', colors='green')
    ax_cp.invert_yaxis()
    ax_cp.yaxis.set_label_position('left')
    ax_cp.yaxis.tick_left()
    ax_cp.spines['right'].set_visible(False)
    
    # Arreglar ejes de los twinx
    ax00.set_yscale('symlog')
    ax01.set_yscale('symlog')
    #separar ejes secundarios y poner del mismo color que los puntos
    ax00.spines['right'].set_position(('outward', 30))
    ax01.spines['right'].set_position(('outward', 90))
    ax00.spines['right'].set_color('blue')
    ax01.spines['right'].set_color('red')
    ax00.tick_params(axis='y', colors='blue')
    ax01.tick_params(axis='y', colors='red')
    ax[0].set_xlabel('x')
    ax[0].set_ylabel('z')
    ax00.set_ylabel(features[0], color='blue')
    ax01.set_ylabel(features[1], color='red')
    ax[0].set_title(f'Case {case}')

    ax[1].scatter(
        x, z, c='black', s=1, alpha=0.7)
    ax[1].set_xlabel('x')
    ax[1].set_ylabel('z')
    ax[1].set_ylim(z.min() - 0.1, z.max() + 0.1)
    ax10 = ax[1].twinx()
    ax10.scatter(
        x, cp, c=clusters, s=markersize_dcp, alpha=0.7, cmap='viridis')
    ax10.set_ylabel('cP')
    ax10.invert_yaxis()

    fig.suptitle(f'Case {case} - features: {features}')
    # bajar título para que no se solape con los ejes
    fig.subplots_adjust(top=0.9)
    if bool(save):
        fig.savefig(
            os.path.join(
                './prueba.png'
                )
            )
        plt.close('all')
    else:
        return fig

In [ ]:
fig = plot_case_0(case=852, vars=['cp', 'cp2'], df=df_data_complete, save=False)